# Synthetic financial time series with a conditional WGAN-GP

This notebook walks through the Tempora pipeline: price data, the return transform, training the Wasserstein GAN with gradient penalty, and the checks that tell you whether synthetic returns are realistic and useful for risk.

It uses the bundled NumPy engine (no GPU or deep-learning framework needed). Training takes about a minute on a laptop CPU.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempora as tp
from tempora.transforms import ReturnTransform, log_returns
from tempora import metrics as M

INK, ORANGE = "#1B2A41", "#F0622E"
print("tempora", tp.__version__)

## 1. Data

By default a **benchmark market** simulated with GJR-GARCH(1,1) and Student-t shocks: it has the stylized facts of real equity returns and a known data-generating process. To use real prices instead:

```python
prices = tp.download("AAPL", start="2010-01-01")     # needs: pip install yfinance
# or
prices = tp.load_prices("data/raw/AAPL.csv")
```

In [ ]:
prices = tp.load_prices(ROOT / "data" / "raw" / "benchmark_prices.csv")
r = log_returns(prices.to_numpy())
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
prices.plot(ax=ax[0], color=INK, title="Price")
ax[1].plot(prices.index[1:], r, color=INK, lw=.6); ax[1].set_title("Daily log returns")
plt.tight_layout()

## 2. The return transform

Standardise, apply the inverse Lambert W transform to make heavy tails close to Gaussian, standardise again. The inverse is exact.

In [ ]:
tr = ReturnTransform().fit(r)
z = tr.transform(r)
print("kurtosis before", round(pd.Series(r).kurt(), 2), "after", round(pd.Series(z).kurt(), 2))
print("round-trip error", np.abs(tr.inverse(z) - r).max())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(r, 120, color=INK); ax[0].set_title("Returns")
ax[1].hist(z, 120, color=ORANGE); ax[1].set_title("Transformed")
plt.tight_layout()

## 3. Train and evaluate

In [ ]:
cfg = tp.ExperimentConfig(gan=tp.GANConfig(iterations=4000))
exp = tp.run(prices, cfg)
res = exp.results
pd.DataFrame(res["scorecard"])

In [ ]:
h = exp.model.history
fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
ax[0].plot(pd.Series(h.wasserstein).rolling(50).mean(), color=ORANGE); ax[0].set_title("Wasserstein estimate")
ax[1].plot(pd.Series(h.grad_norm).rolling(50).mean(), color=INK); ax[1].axhline(1, ls="--", c="grey"); ax[1].set_title("Critic gradient norm")
ax[2].plot(h.lr_d, color=INK, label="critic"); ax[2].plot(h.lr_g, color=ORANGE, label="generator"); ax[2].legend(); ax[2].set_title("Learning rates")
plt.tight_layout()

## 4. Stylized facts

In [ ]:
f = res["facts"]
fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
for a, key, title in zip(ax, ["acf_returns", "acf_abs", "leverage"], ["Return ACF", "|return| ACF (volatility clustering)", "Leverage effect"]):
    a.plot(f["lags"], f["real"][key], "o-", color=INK, label="real")
    a.plot(f["lags"], f["synthetic"][key], "o-", color=ORANGE, label="synthetic")
    a.axhline(0, c="grey", lw=.8); a.set_title(title); a.legend()
plt.tight_layout()
pd.DataFrame({k: [f["real"][k], f["synthetic"][k]] for k in ["std", "skew", "kurtosis", "var99", "es99"]}, index=["real", "synthetic"])

## 5. Scenarios and risk backtests

In [ ]:
c = res["contexts"][len(res["contexts"]) // 2]
paths = np.array(c["paths"]); base = c["cond"][-1]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(-len(c["history"]) - 2, -2), c["history"], color=INK)
ax.plot(range(-2, 1), c["cond"], color=INK, lw=3)
for p in paths: ax.plot(range(0, 21), [base] + list(p), color=ORANGE, alpha=.25, lw=1)
ax.fill_between(range(1, 21), c["q"]["0.05"], c["q"]["0.95"], color=ORANGE, alpha=.15)
ax.plot(range(0, 21), [base] + c["real"], color=INK, lw=2.5, ls="--", label="what happened")
ax.legend(); ax.set_title(f"Scenarios from {c['start']}")

In [ ]:
cov = res["coverage"]
print("Interval coverage:", {k: round(v, 3) for k, v in cov["coverage"].items()})
print("95% VaR breach rate:", round(res["var_backtest"]["rate"], 3))
print("Discriminative accuracy:", round(res["discriminative"]["accuracy"], 3))

## 6. Export for the dashboard

In [ ]:
written = tp.export(exp, ROOT / "data" / "processed", ROOT / "models", ROOT / "dashboard" / "data" / "tempora_data.js")
[w.name for w in written]